## 연습 1: 카탈로그 구조 설정

당신은 **Lakeside University**의 데이터 엔지니어입니다. Lakeside University는 학사 운영을 디지털화하는 중형 기관입니다. 팀은 학생 기록, 강좌 카탈로그 및 등록 데이터를 관리하기 위해 Azure Databricks 및 Unity Catalog에서 데이터 플랫폼을 구축하고 있습니다.

이 연습에서는 최상위 네임스페이스 구조를 생성합니다: 하나의 `카탈로그`와 **메달리온 아키텍처** 네이밍 패턴을 따르는 3개의 `스키마`.

생성할 Unity Catalog 구조는 다음과 같습니다:

```txt

Unity Catalog
└── edu_dev  [카탈로그]
    │   태그: environment=development | university=lakeside | data_classification=internal
    │
    ├── bronze  [스키마]
    │   └── raw_files/  [관리형 볼륨]
    │       └── students.csv
    │
    ├── silver  [스키마]
    │   ├── students  [테이블]
    │   │   ├── student_id       BIGINT  NOT NULL  ← PK
    │   │   ├── first_name       STRING
    │   │   ├── last_name        STRING
    │   │   ├── email            STRING
    │   │   ├── enrollment_year  INT
    │   │   ├── program          STRING
    │   │   └── phone_number     STRING  (연습 6에서 추가됨)
    │   │
    │   ├── courses  [테이블]
    │   │   ├── course_id    BIGINT  NOT NULL  ← PK
    │   │   ├── course_name  STRING
    │   │   ├── department   STRING
    │   │   └── credits      INT
    │   │
    │   ├── enrollments  [테이블]
    │   │   ├── enrollment_id  BIGINT  NOT NULL  ← PK
    │   │   ├── student_id     BIGINT            → FK → students.student_id
    │   │   ├── course_id      BIGINT            → FK → courses.course_id
    │   │   ├── semester       STRING
    │   │   └── grade          DECIMAL(4,2)
    │   │
    │   ├── vw_student_enrollments  [뷰]
    │   │   └── students + courses + enrollments 조인
    │   │
    │   └── get_grade_classification(grade)  [함수]
    │       └── STRING 반환  (A / B / C / D / F)
    │
    └── gold  [스키마]
        └── vw_department_enrollment_stats  [구체화된 뷰]
            └── enrollments + courses 집계
                (department, total_enrollments, avg_grade, distinct_students)
```

### 작업 1.1 — 개발 카탈로그 생성

`edu_dev`라는 카탈로그를 설명 주석과 함께 생성하세요. 이 카탈로그는 Lakeside University의 데이터 플랫폼의 **개발 환경** 역할을 합니다.

**네이밍 규칙 상기:**
- 소문자 및 언더스코어 사용
- 마침표, 공백, 슬래시 또는 제어 문자 없음

> 🤖 **Genie Code 팁:** Genie Code에 질문하세요:
> *"Databricks SQL에서 주석이 있는 Unity Catalog 카탈로그를 생성하는 방법은?"*

**힌트:** SQL 명령 `CREATE CATALOG`을 사용하세요.

In [ ]:
# 기본 저장소가 활성화되지 않았으므로 기본 카탈로그의 루트에서 저장소 경로를 상속합니다.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_ws_*")]
adb_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {adb_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"저장소 루트: {storage_root}")

In [ ]:
%sql
-- TODO: 설명과 함께 edu_dev 카탈로그를 생성하세요

### 작업 1.2 — 메달리온 스키마 생성

`edu_dev` 카탈로그 내에서 **메달리온 아키텍처**를 구현하는 3개의 스키마를 생성하세요:

| 스키마 | 목적 |
|--------|---------|
| `bronze` | 원본 수집 데이터 — 수정되지 않은 소스 파일 |
| `silver` | 정제, 검증 및 강화된 데이터 |
| `gold` | 집계되고 분석 준비가 완료된 데이터 |

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Databricks Unity Catalog 카탈로그에서 SQL을 사용하여 여러 스키마를 생성하는 방법을 보여주세요."*

**힌트:** 각 스키마에 대해 `CREATE SCHEMA IF NOT EXISTS catalog.schema_name`을 사용하세요.

In [ ]:
%sql
-- TODO: edu_dev 내에 bronze, silver, gold 스키마를 생성하세요

## 연습 2: 제약 조건이 있는 테이블 생성

카탈로그 구조가 준비되었으므로 `silver` 스키마의 Lakeside University 학사 데이터에 대한 핵심 테이블을 생성하세요. **기본 키** 및 **외래 키** 제약 조건을 정의하여 데이터 모델을 문서화합니다. Unity Catalog에서 이러한 제약 조건은 정보 제공용입니다 — 쿼리 최적화에 도움이 되지만 쓰기 시간에 참조 무결성을 적용하지는 않습니다.

### 작업 2.1 — `students` 테이블 생성

학생 기록을 저장할 관리형 Delta 테이블 `edu_dev.silver.students`를 다음 열로 생성하세요:

| 열 | 유형 | 설명 |
|--------|------|-------------|
| `student_id` | BIGINT | 고유 학생 식별자 (기본 키) |
| `first_name` | STRING | 학생 이름 |
| `last_name` | STRING | 학생 성 |
| `email` | STRING | 대학 이메일 주소 |
| `enrollment_year` | INT | 학생이 등록한 연도 |
| `program` | STRING | 학위 프로그램 |

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Unity Catalog에서 기본 키 제약 조건이 있는 관리형 Delta 테이블을 생성하는 방법은?"*

**힌트:** 열 목록 끝에 `CONSTRAINT students_pk PRIMARY KEY (student_id)`을 추가하세요.

In [ ]:
%sql
-- TODO: 기본 키 제약 조건으로 edu_dev.silver.students를 생성하세요

### 작업 2.2 — `courses` 테이블 생성

다음 열로 `edu_dev.silver.courses`를 생성하세요:

| 열 | 유형 | 설명 |
|--------|------|-------------|
| `course_id` | BIGINT | 고유 강좌 식별자 (기본 키) |
| `course_name` | STRING | 강좌의 전체 이름 |
| `department` | STRING | 강좌를 제공하는 학부 |
| `credits` | INT | 학점 수 |

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Databricks Unity Catalog에서 기본 키 제약 조건이 있는 courses 테이블을 위한 CREATE TABLE 문을 작성하세요."*

In [ ]:
%sql
-- TODO: 기본 키 제약 조건으로 edu_dev.silver.courses를 생성하세요

### 작업 2.3 — 외래 키가 있는 `enrollments` 테이블 생성

어떤 학생이 어떤 강좌에 등록했는지를 기록하는 `edu_dev.silver.enrollments`를 생성하세요. 이 테이블은 외래 키 제약 조건을 통해 `students` 및 `courses`를 참조해야 합니다.

| 열 | 유형 | 설명 |
|--------|------|-------------|
| `enrollment_id` | BIGINT | 고유 등록 기록 (기본 키) |
| `student_id` | BIGINT | `students.student_id`를 참조 |
| `course_id` | BIGINT | `courses.course_id`를 참조 |
| `semester` | STRING | 학기 (예: `Spring 2023`) |
| `grade` | DECIMAL(4,2) | 0.0–10.0 규모의 수치 성적 |

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Databricks Unity Catalog에서 테이블을 생성할 때 외래 키 제약 조건을 추가하여 다른 테이블을 참조하는 방법은?"*

**힌트:** 각 외래 키에 대해 `CONSTRAINT <이름> FOREIGN KEY (<열>) REFERENCES <카탈로그>.<스키마>.<테이블>(<열>)`을 사용하세요.

In [ ]:
%sql
-- TODO: 기본 키 및 두 개의 외래 키 제약 조건으로 edu_dev.silver.enrollments를 생성하세요

### 샘플 데이터 — 이 셀을 실행하세요

다음 셀은 3개의 테이블에 샘플 데이터를 삽입합니다. 연습 3을 계속하기 전에 **수정 없이 실행하세요**.

In [ ]:
%sql
-- 학생 채우기
INSERT INTO edu_dev.silver.students VALUES
(1001, 'Emma',   'Watson',    'emma.watson@lakeside.edu',    2022, 'Computer Science'),
(1002, 'Liam',   'Ahmed',     'liam.ahmed@lakeside.edu',     2023, 'Data Science'),
(1003, 'Sofia',  'Chen',      'sofia.chen@lakeside.edu',     2021, 'Mathematics'),
(1004, 'James',  'Okonkwo',   'james.okonkwo@lakeside.edu',  2022, 'Computer Science'),
(1005, 'Aisha',  'Patel',     'aisha.patel@lakeside.edu',    2023, 'Data Science'),
(1006, 'Noah',   'Yamamoto',  'noah.yamamoto@lakeside.edu',  2020, 'Mathematics'),
(1007, 'Olivia', 'Kowalski',  'olivia.kowalski@lakeside.edu',2021, 'Computer Science'),
(1008, 'Carlos', 'Reyes',     'carlos.reyes@lakeside.edu',   2022, 'Data Science'),
(1009, 'Mei',    'Lindqvist', 'mei.lindqvist@lakeside.edu',  2023, 'Mathematics'),
(1010, 'Jake',   'OBrien',    'jake.obrien@lakeside.edu',    2020, 'Computer Science');

-- 강좌 채우기
INSERT INTO edu_dev.silver.courses VALUES
(101, 'Introduction to Programming',   'Computer Science', 3),
(102, 'Data Structures',               'Computer Science', 4),
(103, 'Statistics for Data Science',   'Data Science',     3),
(104, 'Machine Learning Fundamentals', 'Data Science',     4),
(105, 'Calculus I',                    'Mathematics',      4),
(106, 'Linear Algebra',                'Mathematics',      3);

-- 등록 채우기
INSERT INTO edu_dev.silver.enrollments VALUES
(1,  1001, 101, 'Spring 2023', 8.50),
(2,  1001, 102, 'Fall 2023',   7.80),
(3,  1002, 103, 'Spring 2023', 9.20),
(4,  1002, 104, 'Fall 2023',   6.50),
(5,  1003, 105, 'Spring 2022', 7.00),
(6,  1003, 106, 'Fall 2022',   8.00),
(7,  1004, 101, 'Fall 2022',   6.80),
(8,  1005, 103, 'Spring 2023', 9.00),
(9,  1006, 105, 'Fall 2020',   7.50),
(10, 1007, 102, 'Spring 2022', 8.20),
(11, 1008, 104, 'Fall 2023',   7.30),
(12, 1009, 106, 'Spring 2023', 8.80),
(13, 1010, 101, 'Fall 2021',   5.50),
(14, 1004, 102, 'Spring 2023', 7.10),
(15, 1005, 104, 'Fall 2023',   8.90);

## 연습 3: 뷰 생성

뷰는 테이블 위에 가상 계층을 제공하여 조인된 데이터에 대한 접근을 단순화하고 비용이 많이 드는 집계를 사전 계산합니다. 이 연습에서는 조합된 등록 기록에 대한 **표준 뷰**와 Lakeside University 분석 팀에 사전 집계된 학부 통계를 제공하는 `gold` 스키마의 **구체화된 뷰**를 생성합니다.

### 작업 3.1 — 표준 뷰 생성

학생 등록 기록의 결합되고 읽기 쉬운 뷰를 제시하는 `edu_dev.silver.vw_student_enrollments` 뷰를 생성하세요. 다음을 포함해야 합니다:

- 학생 전체 이름 (`first_name`과 `last_name`을 공백으로 연결)
- 학생 `email`
- `course_name`
- `department`
- `semester`
- `grade`

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Azure Databricks에서 세 개의 테이블을 조인하는 SQL 뷰를 작성하세요: students, courses, enrollments"*

**힌트:** 이름을 결합하기 위해 `CONCAT(first_name, ' ', last_name)` 또는 `||` 연산자를 사용하세요. 3개의 silver 테이블을 모두 조인해야 합니다.

In [ ]:
%sql
-- TODO: edu_dev.silver.vw_student_enrollments 뷰를 생성하세요
-- students, courses, enrollments를 조인하세요; full_name, email, course_name, department, semester, grade 포함

### 작업 3.2 — 학부 통계에 대한 구체화된 뷰 생성

각 학부별로 다음을 사전 계산하는 구체화된 뷰 `edu_dev.gold.vw_department_enrollment_stats`를 생성하세요:

| 출력 열 | 설명 |
|--------------|-------------|
| `department` | 학부명 |
| `total_enrollments` | 총 등록 기록 수 |
| `avg_grade` | 평균 성적, 소수점 2자리로 반올림 |
| `distinct_students` | 등록한 고유 학생 수 |

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Azure Databricks에서 두 개의 조인된 테이블의 데이터를 집계하는 구체화된 뷰를 생성하기 위한 문법은?"*

**힌트:** `CREATE MATERIALIZED VIEW`를 사용하고 `enrollments`를 `courses`와 조인한 다음 `COUNT()`, `ROUND(AVG(...), 2)`, `COUNT(DISTINCT ...)`를 사용하세요.

> [!NOTE]
> 구체화된 뷰는 실행할 SQL 웨어하우스가 필요합니다. 이 셀이 실패하면 노트북 상단의 컴퓨팅 선택기를 검사하고 **서버리스** 컴퓨팅에서 **서버리스 SQL 웨어하우스**로 전환하세요.

In [ ]:
%sql
-- TODO: edu_dev.gold.vw_department_enrollment_stats 구체화된 뷰를 생성하세요

**✅ 구체화된 뷰 테스트** — 다음 쿼리를 실행하여 학부별 집계를 반환하는지 확인하세요:

각 학부에 대해 `total_enrollments`, `avg_grade`, `distinct_students` 열이 있는 한 행씩 나타나야 합니다.

In [ ]:
%sql
SELECT * FROM edu_dev.gold.vw_department_enrollment_stats;

## 연습 4: 볼륨 생성 및 파일 읽기

볼륨은 Unity Catalog 거버넌스를 구조화된 테이블뿐만 아니라 파일로 확장합니다. 이 연습에서는 `bronze` 스키마에서 **관리형 볼륨**을 생성하여 원본 CSV 파일의 랜딩 영역으로 사용합니다 — Lakeside University의 외부 학생 정보 시스템에서 도착하는 파일을 시뮬레이션합니다.

### 작업 4.1 — 관리형 볼륨 생성

`edu_dev.bronze` 스키마에서 `raw_files`라는 관리형 볼륨을 생성하세요. 이는 원본 데이터 파일의 랜딩 영역으로 사용됩니다.

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Unity Catalog에서 SQL 문을 사용하여 관리형 볼륨을 생성하는 방법은?"*

**힌트:** `CREATE VOLUME IF NOT EXISTS <카탈로그>.<스키마>.<볼륨>`을 사용하세요.

In [ ]:
%sql
-- TODO: edu_dev.bronze에서 raw_files 관리형 볼륨을 생성하세요

### 작업 4.2 — 볼륨에 CSV 파일 쓰기

다음 Python 셀은 샘플 학생 CSV 파일을 새 볼륨에 직접 씁니다. **수정 없이 실행하세요.**

In [ ]:
# 샘플 학생 CSV 파일을 볼륨에 쓰세요 — 이 셀을 수정 없이 실행하세요
csv_content = """student_id,first_name,last_name,email,enrollment_year,program
1001,Emma,Watson,emma.watson@lakeside.edu,2022,Computer Science
1002,Liam,Ahmed,liam.ahmed@lakeside.edu,2023,Data Science
1003,Sofia,Chen,sofia.chen@lakeside.edu,2021,Mathematics
1004,James,Okonkwo,james.okonkwo@lakeside.edu,2022,Computer Science
1005,Aisha,Patel,aisha.patel@lakeside.edu,2023,Data Science
1006,Noah,Yamamoto,noah.yamamoto@lakeside.edu,2020,Mathematics
1007,Olivia,Kowalski,olivia.kowalski@lakeside.edu,2021,Computer Science
1008,Carlos,Reyes,carlos.reyes@lakeside.edu,2022,Data Science
1009,Mei,Lindqvist,mei.lindqvist@lakeside.edu,2023,Mathematics
1010,Jake,OBrien,jake.obrien@lakeside.edu,2020,Computer Science"""

dbutils.fs.put("/Volumes/edu_dev/bronze/raw_files/students.csv", csv_content, overwrite=True)
print("students.csv를 /Volumes/edu_dev/bronze/raw_files/에 썼습니다")

### 작업 4.3 — 볼륨의 CSV를 DataFrame으로 읽기

방금 볼륨에 배치한 CSV 파일을 Spark DataFrame으로 읽으세요. 파일은 `/Volumes/edu_dev/bronze/raw_files/students.csv`에 있습니다.

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Unity Catalog 볼륨 경로에서 헤더 행이 있는 CSV 파일을 PySpark에서 읽는 방법은?"*

**힌트:** `spark.read.csv(path, header=True, inferSchema=True)`를 사용한 다음 결과에 대해 `.display()` 또는 `.show()`를 호출하세요.

In [ ]:
# TODO: /Volumes/edu_dev/bronze/raw_files/students.csv를 Spark DataFrame으로 읽고 표시하세요

## 연습 5: 재사용 가능한 SQL 함수 생성

Unity Catalog를 사용하면 **거버넌스된, 재사용 가능한 함수**를 생성할 수 있으므로 비즈니스 로직을 캡슐화할 수 있습니다. 모든 쿼리에서 성적 변환 논리를 반복하는 대신 한 번 SQL 함수로 작성하고 작업 공간의 모든 노트북 또는 쿼리에서 호출하세요.

### 작업 5.1 — 성적 분류 함수 생성

Lakeside University에서 사용하는 척도에 따라 `grade DECIMAL(4,2)`를 수락하고 `STRING` 문자 분류를 반환하는 SQL 스칼라 함수 `edu_dev.silver.get_grade_classification`을 생성하세요:

| 성적 범위 | 분류 |
|-------------|----------------|
| ≥ 8.5 | `'A'` |
| ≥ 7.0 | `'B'` |
| ≥ 5.5 | `'C'` |
| ≥ 4.0 | `'D'` |
| < 4.0 | `'F'` |

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Unity Catalog에서 CASE 식을 사용하여 수치 점수를 기반으로 문자 성적을 반환하는 SQL 스칼라 함수를 생성하는 방법은?"*

**힌트:** `CREATE FUNCTION ... RETURNS STRING RETURN CASE WHEN ... END`를 사용하세요.

In [ ]:
%sql
-- TODO: edu_dev.silver.get_grade_classification 함수를 생성하세요

### 작업 5.2 — 함수 테스트

`edu_dev.silver.enrollments`를 쿼리하고 새 함수를 적용하여 각 행에 대한 `grade_classification` 열을 생성하세요. 다음을 포함하세요: `enrollment_id`, `student_id`, `course_id`, `semester`, `grade`, `grade_classification`.

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Azure Databricks의 SELECT 문에서 Unity Catalog 사용자 정의 함수를 호출하는 방법은?"*

In [ ]:
%sql
-- TODO: edu_dev.silver.enrollments를 쿼리하고 grade 열에 get_grade_classification을 호출하세요

## 연습 6: DDL 작업

Unity Catalog 개체는 요구 사항 변화에 따라 시간이 지남에 따라 발전합니다. 이 연습에서는 `ALTER` 문을 사용합니다:
1. 새 열을 사용하여 `students` 테이블을 확장합니다.
2. 거버넌스 **태그**를 `edu_dev` 카탈로그에 적용합니다.

### 작업 6.1 — students 테이블에 열 추가

학생 등록 시스템이 전화번호를 캡처하도록 업데이트되었습니다. `STRING` 유형의 null 허용 `phone_number` 열을 `edu_dev.silver.students`에 추가하세요.

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Azure Databricks에서 SQL을 사용하여 기존 Delta 테이블에 새 null 허용 열을 추가하는 방법은?"*

**힌트:** `ALTER TABLE`을 사용하세요.

In [ ]:
%sql
-- TODO: edu_dev.silver.students에 phone_number (STRING) 열을 추가하세요

### 작업 6.2 — 카탈로그에 태그 적용

거버넌스 및 발견성을 지원하기 위해 `edu_dev` 카탈로그에 다음 태그를 적용하세요:

| 태그 키 | 태그 값 |
|---------|----------|
| `environment` | `'development'` |
| `university` | `'lakeside'` |
| `data_classification` | `'internal'` |

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Databricks에서 ALTER CATALOG 문을 사용하여 Unity Catalog 카탈로그에 메타데이터 태그를 설정하는 방법은?"*

In [ ]:
%sql
-- TODO: ALTER CATALOG ... SET TAGS를 사용하여 3개 태그를 edu_dev 카탈로그에 적용하세요

### 작업 6.3 — 작업 확인

다음 셀을 실행하여 카탈로그 구조, 테이블 스키마 및 태그가 모두 준비되어 있는지 확인합니다. 출력을 이 랩 전체에서 생성한 내용과 비교하세요.

In [ ]:
%sql
-- silver 스키마의 모든 테이블 및 뷰 표시
SHOW TABLES IN edu_dev.silver;

In [ ]:
%sql
-- students 테이블 설명 — 새 phone_number 열이 있는지 확인
DESCRIBE TABLE edu_dev.silver.students;

In [ ]:
%sql
-- 카탈로그 세부 정보 표시
DESCRIBE CATALOG EXTENDED edu_dev;

In [ ]:
%sql 
-- 정보 스키마를 통한 태그 쿼리 (구조화된 방식)
SELECT *
FROM system.information_schema.catalog_tags
WHERE catalog_name = 'edu_dev';

---

## 다음 단계

모든 노트북 연습을 완료했습니다. **랩 지침**으로 돌아가서 다음을 계속하세요:

- **연습**: AI/BI Genie Space 구성 (선택 사항)